# v8 — Forzar detección de melanoma

**Problema v7:** mel F1=0%, modelo nunca predice melanoma a pesar de oversample ×3 y class_weight=10.

**Dos estrategias en este notebook:**

### Estrategia A — Umbral diferenciado (sin re-entrenar)
Usar el modelo v7 ya entrenado. En lugar de `argmax(probs)`, aplicar:
- Si `p(mel) > umbral_mel` → predecir mel
- El umbral óptimo se busca en validation set

### Estrategia B — Focal Loss con alpha por clase (re-entrenamiento)
Reentrenar con `FocalLossAlpha` donde:
- `alpha[mel] = 0.95` — penalizar fuertemente los mel mal clasificados
- `alpha[nv]  = 0.05` — reducir el peso de nv en la loss
- Resto proporcional al desbalance

Ambas estrategias se evalúan en test y se comparan.


## 1. Entorno y datos

In [1]:
import os, gc
import numpy as np
import tensorflow as tf
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import cv2
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)
import seaborn as sns

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
N_CLASSES   = len(CLASS_NAMES)
MEL_IDX     = CLASS_NAMES.index('mel')
NV_IDX      = CLASS_NAMES.index('nv')
IMG_SIZE    = 224
BATCH_SIZE  = 32

NUMPY_DIR   = '/kaggle/input/datasets/michellercarranza/ham10000-numpy'
MODELS_V7   = '/kaggle/input/datasets/michellercarranza/modelos-v7'
OUTPUT_DIR  = '/kaggle/working/v8'
os.makedirs(OUTPUT_DIR, exist_ok=True)


TensorFlow: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
# Cargar datos
print('Cargando datos...')
X_train = np.load(os.path.join(NUMPY_DIR, 'train_X.npy'))
y_train = np.load(os.path.join(NUMPY_DIR, 'train_y.npy'))
X_val   = np.load(os.path.join(NUMPY_DIR, 'validation_X.npy'))
y_val   = np.load(os.path.join(NUMPY_DIR, 'validation_y.npy'))
X_test  = np.load(os.path.join(NUMPY_DIR, 'test_X.npy'))
y_test  = np.load(os.path.join(NUMPY_DIR, 'test_y.npy'))

y_train_int = y_train
y_val_int   = y_val
y_test_int  = y_test

print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')
print('\nDistribución train:')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_int == i)
    print(f'  {name:6s}: {n:5d}')


Cargando datos...
Train: (9577, 224, 224, 3)  Val: (2492, 224, 224, 3)  Test: (1285, 224, 224, 3)

Distribución train:
  akiec :   315
  bcc   :   487
  bkl   :  1048
  df    :   110
  mel   :  1076
  nv    :  6405
  vasc  :   136


## 2. Hair Removal (igual que v7)

In [3]:
def remove_hair(img_uint8, kernel_size=17, threshold=10):
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_size, kernel_size))
    blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)
    _, mask = cv2.threshold(blackhat, threshold, 255, cv2.THRESH_BINARY)
    kernel_dilate = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.dilate(mask, kernel_dilate, iterations=1)
    img_bgr = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2BGR)
    inpainted_bgr = cv2.inpaint(img_bgr, mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
    return cv2.cvtColor(inpainted_bgr, cv2.COLOR_BGR2RGB)


def preprocess_batch(X_float32):
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    result = np.empty_like(X_float32)
    for i in range(len(X_float32)):
        img_uint8 = np.clip((X_float32[i] * std + mean) * 255, 0, 255).astype(np.uint8)
        img_clean = remove_hair(img_uint8)
        result[i] = (img_clean.astype(np.float32) / 255.0 - mean) / std
    return result


print('Aplicando hair removal...')
CHUNK = 500
X_train_clean = np.empty_like(X_train)
for i in range(0, len(X_train), CHUNK):
    end = min(i + CHUNK, len(X_train))
    X_train_clean[i:end] = preprocess_batch(X_train[i:end])
    if (i // CHUNK + 1) % 5 == 0:
        print(f'  train {end}/{len(X_train)}')
X_val_clean  = preprocess_batch(X_val)
X_test_clean = preprocess_batch(X_test)
del X_train, X_val, X_test
gc.collect()
print('Hair removal completado.')


Aplicando hair removal...
  train 2500/9577
  train 5000/9577
  train 7500/9577
  train 9577/9577
Hair removal completado.


## 3. Oversample mel ×3 + Generador

In [4]:
mel_train_indices = np.where(y_train_int == MEL_IDX)[0]
X_mel_extra = np.tile(X_train_clean[mel_train_indices], (2, 1, 1, 1))
y_mel_extra = np.tile(y_train_int[mel_train_indices], 2)
X_train_aug = np.concatenate([X_train_clean, X_mel_extra], axis=0)
y_train_aug = np.concatenate([y_train_int, y_mel_extra], axis=0)
del X_mel_extra, y_mel_extra
gc.collect()
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]
del perm; gc.collect()

print(f'Train aumentado: {X_train_aug.shape}')
for i, name in enumerate(CLASS_NAMES):
    n = np.sum(y_train_aug == i)
    print(f'  {name:6s}: {n:5d} ({n/len(y_train_aug)*100:.1f}%)')


Train aumentado: (11729, 224, 224, 3)
  akiec :   315 (2.7%)
  bcc   :   487 (4.2%)
  bkl   :  1048 (8.9%)
  df    :   110 (0.9%)
  mel   :  3228 (27.5%)
  nv    :  6405 (54.6%)
  vasc  :   136 (1.2%)


In [5]:
def augment_image(img, is_mel=False):
    img = img.copy()
    mean_v = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std_v  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    if np.random.rand() > 0.5: img = np.fliplr(img)
    if np.random.rand() > 0.5: img = np.flipud(img)
    max_angle = 360 if is_mel else 90
    angle = np.random.uniform(-max_angle, max_angle)
    h, w = img.shape[:2]
    M = cv2.getRotationMatrix2D((w/2, h/2), angle, 1.0)
    img_uint8 = np.clip((img * std_v + mean_v) * 255, 0, 255).astype(np.uint8)
    img_uint8 = cv2.warpAffine(img_uint8, M, (w, h), borderMode=cv2.BORDER_REFLECT)
    img = (img_uint8.astype(np.float32) / 255.0 - mean_v) / std_v
    if is_mel:
        alpha = np.random.uniform(0.85, 1.15)
        beta  = np.random.uniform(-0.05, 0.05)
        img = img * alpha + beta
    return img.astype(np.float32)


class AugmentedGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y, batch_size, mel_idx, augment=True, shuffle=True):
        self.X, self.y = X, y
        self.batch_size = batch_size
        self.mel_idx = mel_idx
        self.augment = augment
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        if shuffle: np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        X_batch = np.empty((len(batch_idx), IMG_SIZE, IMG_SIZE, 3), dtype=np.float32)
        y_batch = self.y[batch_idx]
        for i, j in enumerate(batch_idx):
            img = self.X[j]
            if self.augment:
                img = augment_image(img, is_mel=(self.y[j] == self.mel_idx))
            X_batch[i] = img
        return X_batch, tf.keras.utils.to_categorical(y_batch, N_CLASSES)

    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)


train_gen = AugmentedGenerator(X_train_aug, y_train_aug, BATCH_SIZE, MEL_IDX)
val_gen   = AugmentedGenerator(X_val_clean, y_val_int, BATCH_SIZE, MEL_IDX, augment=False, shuffle=False)
print(f'Generadores listos. Train batches: {len(train_gen)}')


Generadores listos. Train batches: 367


## 4. Estrategia B — Focal Loss con alpha por clase

`alpha` controla el peso de cada clase en la loss:
- `alpha[mel] = 0.95` → loss de mel se multiplica por 0.95/(1-0.95)=19×
- `alpha[nv]  = 0.05` → loss de nv se reduce drásticamente


In [6]:
class FocalLossAlpha(tf.keras.losses.Loss):
    """
    Focal Loss con alpha por clase.
    alpha: lista de pesos por clase [0,1], sum(alpha)=1 no obligatorio.
    gamma: factor de focalización (2.0 estándar)
    """
    def __init__(self, alpha=None, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        if alpha is None:
            self.alpha = tf.ones(N_CLASSES, dtype=tf.float32) / N_CLASSES
        else:
            self.alpha = tf.constant(alpha, dtype=tf.float32)

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        # alpha ponderado por clase verdadera
        alpha_t = tf.reduce_sum(y_true * self.alpha, axis=-1, keepdims=True)
        # probabilidad de la clase verdadera
        pt = tf.reduce_sum(y_true * y_pred, axis=-1, keepdims=True)
        # Focal weight
        focal_weight = alpha_t * tf.pow(1.0 - pt, self.gamma)
        # Cross-entropy por clase
        ce = -y_true * tf.math.log(y_pred)
        loss = focal_weight * tf.reduce_sum(ce, axis=-1, keepdims=True)
        return tf.reduce_mean(loss)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'alpha': self.alpha.numpy().tolist(), 'gamma': self.gamma})
        return cfg


# Alpha por clase: mel=0.95, nv=0.05, resto proporcional al desbalance
counts = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
inv_freq = 1.0 / counts
inv_freq = inv_freq / inv_freq.sum()  # normalizar

alpha_values = inv_freq.copy()
alpha_values[MEL_IDX] = 0.95
alpha_values[NV_IDX]  = 0.05
# Renormalizar el resto
other_idx = [i for i in range(N_CLASSES) if i not in [MEL_IDX, NV_IDX]]
other_sum = alpha_values[other_idx].sum()
scale = (1.0 - 0.95 - 0.05) / other_sum if other_sum > 0 else 1.0
for i in other_idx:
    alpha_values[i] *= scale

print('Alpha por clase:')
for i, name in enumerate(CLASS_NAMES):
    print(f'  {name:6s}: {alpha_values[i]:.4f}')
print(f'  Suma: {alpha_values.sum():.4f}')

focal_loss_alpha = FocalLossAlpha(alpha=alpha_values.tolist(), gamma=2.0)
print('FocalLossAlpha lista.')


Alpha por clase:
  akiec : 0.0000
  bcc   : 0.0000
  bkl   : 0.0000
  df    : 0.0000
  mel   : 0.9500
  nv    : 0.0500
  vasc  : 0.0000
  Suma: 1.0000
FocalLossAlpha lista.


## 5. Construcción del modelo

In [7]:
def build_model():
    base = tf.keras.applications.DenseNet201(
        include_top=False, weights='imagenet',
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = False
    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = base(inputs, training=False)
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu',
                               kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(N_CLASSES, activation='softmax')(x)
    return tf.keras.Model(inputs, outputs), base


model, base_model = build_model()
print(f'Modelo construido. Parámetros: {model.count_params():,}')


74836368/74836368 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Modelo construido. Parámetros: 18,815,559


## 6. Class weights extremos

In [8]:
# Class weights aún más agresivos que v7
class_weights = {}
counts_aug = np.array([np.sum(y_train_aug == i) for i in range(N_CLASSES)], dtype=np.float32)
base_w = counts_aug.max() / counts_aug

for i, name in enumerate(CLASS_NAMES):
    if i == MEL_IDX:
        w = 15.0   # v7 tenía 10, subimos a 15
    elif i == NV_IDX:
        w = 0.2    # v7 tenía 0.3, bajamos a 0.2
    else:
        w = float(base_w[i])
    class_weights[i] = w
    print(f'  {name:6s}: weight = {w:.3f}')


  akiec : weight = 20.333
  bcc   : weight = 13.152
  bkl   : weight = 6.112
  df    : weight = 58.227
  mel   : weight = 15.000
  nv    : weight = 0.200
  vasc  : weight = 47.096


## 7. Entrenamiento (3 fases)

In [9]:
# ── Fase 1: cabeza, 10 épocas, CrossEntropy ──
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['accuracy']
)
cb1 = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v8_phase1_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
print('Fase 1 (10 épocas)...')
h1 = model.fit(train_gen, validation_data=val_gen, epochs=10,
               callbacks=cb1, class_weight=class_weights, verbose=1)
print('Fase 1 completada.')


Fase 1 (10 épocas)...


/usr/local/lib/python3.13/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.2620 - loss: 9.4441
Epoch 1: val_accuracy improved from None to 0.13604, saving model to /kaggle/working/v8/densenet201_v8_phase1_best.keras

Epoch 1: finished saving model to /kaggle/working/v8/densenet201_v8_phase1_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 175s 348ms/step - accuracy: 0.2724 - loss: 7.9948 - val_accuracy: 0.1360 - val_loss: 2.7767 - learning_rate: 0.0010
Epoch 2/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step - accuracy: 0.2849 - loss: 7.0158
Epoch 2: val_accuracy improved from 0.13604 to 0.13804, saving model to /kaggle/working/v8/densenet201_v8_phase1_best.keras

Epoch 2: finished saving model to /kaggle/working/v8/densenet201_v8_phase1_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 98s 268ms/step - accuracy: 0.2895 - loss: 6.8155 - val_accuracy: 0.1380 - val_loss: 2.8756 - learning_rate: 0.0010
Epoch 3/10
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 174ms/step - accuracy: 0.2940 - loss: 6.5257
Epoch 3: val_accuracy improv

In [10]:
# ── Fase 2a: descongelar 25 capas + FocalLossAlpha ──
base_model.trainable = True
for layer in base_model.layers[:-25]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(2e-4),
    loss=focal_loss_alpha,
    metrics=['accuracy']
)
cb2a = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v8_phase2a_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1)
]
print('Fase 2a (15 épocas, FocalLossAlpha)...')
h2a = model.fit(train_gen, validation_data=val_gen, epochs=15,
                callbacks=cb2a, class_weight=class_weights, verbose=1)
print('Fase 2a completada.')


Fase 2a (15 épocas, FocalLossAlpha)...
Epoch 1/15
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 231ms/step - accuracy: 0.4790 - loss: 0.3029
Epoch 1: val_accuracy improved from None to 0.47472, saving model to /kaggle/working/v8/densenet201_v8_phase2a_best.keras

Epoch 1: finished saving model to /kaggle/working/v8/densenet201_v8_phase2a_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 187s 373ms/step - accuracy: 0.5011 - loss: 0.2042 - val_accuracy: 0.4747 - val_loss: 0.0321 - learning_rate: 2.0000e-04
Epoch 2/15
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 146ms/step - accuracy: 0.5397 - loss: 0.1246
Epoch 2: val_accuracy did not improve from 0.47472
367/367 ━━━━━━━━━━━━━━━━━━━━ 64s 175ms/step - accuracy: 0.5417 - loss: 0.1213 - val_accuracy: 0.4687 - val_loss: 0.0305 - learning_rate: 2.0000e-04
Epoch 3/15
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 197ms/step - accuracy: 0.5467 - loss: 0.1119
Epoch 3: val_accuracy improved from 0.47472 to 0.52167, saving model to /kaggle/working/v8/densenet201_v8_phase2a_best.keras

Epoch 3: fini

In [11]:
# ── Fase 2b: descongelar 100 capas ──
for layer in base_model.layers[:-100]:
    layer.trainable = False
for layer in base_model.layers[-100:]:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss=focal_loss_alpha,
    metrics=['accuracy']
)
cb2b = [
    tf.keras.callbacks.ModelCheckpoint(
        os.path.join(OUTPUT_DIR, 'densenet201_v8_phase2b_best.keras'),
        monitor='val_accuracy', save_best_only=True, verbose=1),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=7, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1)
]
print('Fase 2b (20 épocas, FocalLossAlpha)...')
h2b = model.fit(train_gen, validation_data=val_gen, epochs=20,
                callbacks=cb2b, class_weight=class_weights, verbose=1)
print('Fase 2b completada.')


Fase 2b (20 épocas, FocalLossAlpha)...
Epoch 1/20
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 300ms/step - accuracy: 0.6675 - loss: 0.0677
Epoch 1: val_accuracy improved from None to 0.63684, saving model to /kaggle/working/v8/densenet201_v8_phase2b_best.keras

Epoch 1: finished saving model to /kaggle/working/v8/densenet201_v8_phase2b_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 241s 448ms/step - accuracy: 0.6675 - loss: 0.0664 - val_accuracy: 0.6368 - val_loss: 0.0196 - learning_rate: 5.0000e-05
Epoch 2/20
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 178ms/step - accuracy: 0.6780 - loss: 0.0645
Epoch 2: val_accuracy improved from 0.63684 to 0.64807, saving model to /kaggle/working/v8/densenet201_v8_phase2b_best.keras

Epoch 2: finished saving model to /kaggle/working/v8/densenet201_v8_phase2b_best.keras
367/367 ━━━━━━━━━━━━━━━━━━━━ 79s 214ms/step - accuracy: 0.6808 - loss: 0.0613 - val_accuracy: 0.6481 - val_loss: 0.0188 - learning_rate: 5.0000e-05
Epoch 3/20
367/367 ━━━━━━━━━━━━━━━━━━━━ 0s 159ms/step - accuracy:

## 8. Estrategia A — Búsqueda del umbral óptimo para mel

Usando el modelo recién entrenado, buscamos el umbral `t` tal que:
- Si `p(mel) > t` → predecir mel
- Optimizamos `t` para maximizar F1-macro en validation


In [12]:
print('Calculando probabilidades en validation...')
val_probs = model.predict(X_val_clean, batch_size=64, verbose=1)
print(f'val_probs shape: {val_probs.shape}')

# Buscar umbral óptimo
thresholds = np.arange(0.05, 0.60, 0.01)
f1_mel_scores  = []
f1_macro_scores = []

for t in thresholds:
    preds = np.argmax(val_probs, axis=1).copy()
    # Sobrescribir con mel donde p(mel) > t
    preds[val_probs[:, MEL_IDX] > t] = MEL_IDX
    f1_mel = f1_score(y_val_int == MEL_IDX,
                      preds == MEL_IDX, zero_division=0)
    f1_mac = f1_score(y_val_int, preds, average='macro', zero_division=0)
    f1_mel_scores.append(f1_mel)
    f1_macro_scores.append(f1_mac)

# Umbral que maximiza F1-macro
best_t_macro = thresholds[np.argmax(f1_macro_scores)]
best_t_mel   = thresholds[np.argmax(f1_mel_scores)]

print(f'\nUmbral óptimo para F1-macro: {best_t_macro:.2f}  (F1={max(f1_macro_scores):.4f})')
print(f'Umbral óptimo para F1-mel:   {best_t_mel:.2f}   (F1-mel={max(f1_mel_scores):.4f})')

# Graficar
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_macro_scores, label='F1-macro', color='#1a6e35', linewidth=2)
ax.plot(thresholds, f1_mel_scores,   label='F1-mel',   color='#c85a08', linewidth=2)
ax.axvline(best_t_macro, color='#1a6e35', linestyle='--', alpha=0.7,
           label=f'Mejor macro t={best_t_macro:.2f}')
ax.axvline(best_t_mel, color='#c85a08', linestyle='--', alpha=0.7,
           label=f'Mejor mel t={best_t_mel:.2f}')
ax.set_xlabel('Umbral para mel')
ax.set_ylabel('F1 Score')
ax.set_title('Búsqueda de umbral óptimo (val set)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threshold_search.png'), dpi=150)
plt.show()
print('Guardada: threshold_search.png')


Calculando probabilidades en validation...
39/39 ━━━━━━━━━━━━━━━━━━━━ 65s 989ms/step
val_probs shape: (2492, 7)

Umbral óptimo para F1-macro: 0.50  (F1=0.1922)
Umbral óptimo para F1-mel:   0.50   (F1-mel=0.4766)
Guardada: threshold_search.png


## 9. Evaluación Estrategia A en test set

In [13]:
print('Calculando probabilidades en test...')
test_probs = model.predict(X_test_clean, batch_size=64, verbose=1)

# Aplicar umbral óptimo (usar best_t_macro para balance general)
THRESHOLD_MEL = best_t_macro
y_pred_A = np.argmax(test_probs, axis=1).copy()
y_pred_A[test_probs[:, MEL_IDX] > THRESHOLD_MEL] = MEL_IDX

print(f'\n=== ESTRATEGIA A — Umbral mel > {THRESHOLD_MEL:.2f} ===')
print(classification_report(y_test_int, y_pred_A,
                             target_names=CLASS_NAMES, zero_division=0))

acc_A = accuracy_score(y_test_int, y_pred_A)
f1_A  = f1_score(y_test_int, y_pred_A, average='macro', zero_division=0)
f1_mel_A = f1_score((y_test_int==MEL_IDX), (y_pred_A==MEL_IDX), zero_division=0)
print(f'Accuracy: {acc_A:.4f} | F1-macro: {f1_A:.4f} | F1-mel: {f1_mel_A:.4f}')


Calculando probabilidades en test...
21/21 ━━━━━━━━━━━━━━━━━━━━ 27s 1s/step 

=== ESTRATEGIA A — Umbral mel > 0.50 ===
              precision    recall  f1-score   support

       akiec       0.00      0.00      0.00        42
         bcc       0.00      0.00      0.00        67
         bkl       0.00      0.00      0.00       142
          df       0.00      0.00      0.00        14
         mel       0.32      0.99      0.48       144
          nv       0.88      0.86      0.87       858
        vasc       0.00      0.00      0.00        18

    accuracy                           0.69      1285
   macro avg       0.17      0.26      0.19      1285
weighted avg       0.63      0.69      0.64      1285

Accuracy: 0.6864 | F1-macro: 0.1933 | F1-mel: 0.4797


## 10. Evaluación Estrategia B — argmax directo (con FocalLossAlpha)

In [14]:
y_pred_B = np.argmax(test_probs, axis=1)

print('=== ESTRATEGIA B — argmax directo (FocalLossAlpha) ===')
print(classification_report(y_test_int, y_pred_B,
                             target_names=CLASS_NAMES, zero_division=0))

acc_B = accuracy_score(y_test_int, y_pred_B)
f1_B  = f1_score(y_test_int, y_pred_B, average='macro', zero_division=0)
f1_mel_B = f1_score((y_test_int==MEL_IDX), (y_pred_B==MEL_IDX), zero_division=0)
print(f'Accuracy: {acc_B:.4f} | F1-macro: {f1_B:.4f} | F1-mel: {f1_mel_B:.4f}')


=== ESTRATEGIA B — argmax directo (FocalLossAlpha) ===
              precision    recall  f1-score   support

       akiec       0.00      0.00      0.00        42
         bcc       0.00      0.00      0.00        67
         bkl       0.00      0.00      0.00       142
          df       0.00      0.00      0.00        14
         mel       0.32      0.99      0.48       144
          nv       0.88      0.86      0.87       858
        vasc       0.00      0.00      0.00        18

    accuracy                           0.69      1285
   macro avg       0.17      0.26      0.19      1285
weighted avg       0.63      0.69      0.64      1285

Accuracy: 0.6864 | F1-macro: 0.1933 | F1-mel: 0.4797


## 11. Combinación A+B — FocalLossAlpha + umbral

In [15]:
# Buscar umbral óptimo también en test
# (en producción real esto se haría solo en val, aquí es para diagnóstico)
y_pred_AB = np.argmax(test_probs, axis=1).copy()
y_pred_AB[test_probs[:, MEL_IDX] > THRESHOLD_MEL] = MEL_IDX

# Es igual a A porque el threshold se calculó sobre val y se aplica al mismo modelo
# La diferencia real está en si B ya predice algo de mel sin threshold
mel_pred_count_B = np.sum(y_pred_B == MEL_IDX)
mel_pred_count_A = np.sum(y_pred_A == MEL_IDX)
print(f'Predicciones mel sin umbral (B):    {mel_pred_count_B}')
print(f'Predicciones mel con umbral (A/AB): {mel_pred_count_A}')
print(f'Muestras mel reales en test:        {np.sum(y_test_int == MEL_IDX)}')


Predicciones mel sin umbral (B):    448
Predicciones mel con umbral (A/AB): 448
Muestras mel reales en test:        144


## 12. Comparativa visual: matrices de confusión

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, y_pred, title in [
    (axes[0], y_pred_B, f'Estrategia B — FocalLossAlpha\nAcc={acc_B:.3f} F1={f1_B:.3f} F1-mel={f1_mel_B:.3f}'),
    (axes[1], y_pred_A, f'Estrategia A+B — FocalLossAlpha + Umbral {THRESHOLD_MEL:.2f}\nAcc={acc_A:.3f} F1={f1_A:.3f} F1-mel={f1_mel_A:.3f}'),
]:
    cm = confusion_matrix(y_test_int, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='OrRd',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=0.5, ax=ax)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel('Clase real')
    ax.set_xlabel('Clase predicha')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrices_v8.png'), dpi=130, bbox_inches='tight')
plt.show()
print('Guardada: confusion_matrices_v8.png')


Guardada: confusion_matrices_v8.png


## 13. Resumen comparativo

In [17]:
print('=' * 65)
print('COMPARATIVA DE VERSIONES')
print('=' * 65)
print(f'{"Versión":20s} {"Accuracy":>10} {"F1-macro":>10} {"F1-mel":>10}')
print('-' * 55)
print(f'{"v5 (baseline)":20s} {0.6879:>10.4f} {0.4362:>10.4f} {0.0000:>10.4f}')
print(f'{"v7 (hair+oversample)":20s} {0.80:>10.4f} {"?":>10} {0.0000:>10.4f}')
print(f'{"v8-B (FocalAlpha)":20s} {acc_B:>10.4f} {f1_B:>10.4f} {f1_mel_B:>10.4f}')
print(f'{"v8-A (FocalAlpha+t)":20s} {acc_A:>10.4f} {f1_A:>10.4f} {f1_mel_A:>10.4f}')
print('=' * 65)

print('\nArchivos generados:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    print(f'  {f:50s} {os.path.getsize(fp)/1e6:.1f} MB')


COMPARATIVA DE VERSIONES
Versión                Accuracy   F1-macro     F1-mel
-------------------------------------------------------
v5 (baseline)            0.6879     0.4362     0.0000
v7 (hair+oversample)     0.8000          ?     0.0000
v8-B (FocalAlpha)        0.6864     0.1933     0.4797
v8-A (FocalAlpha+t)      0.6864     0.1933     0.4797

Archivos generados:
  confusion_matrices_v8.png                          0.1 MB
  densenet201_v8_phase1_best.keras                   81.7 MB
  densenet201_v8_phase2a_best.keras                  88.7 MB
  densenet201_v8_phase2b_best.keras                  110.4 MB
  threshold_search.png                               0.1 MB
